# User Experience para ciência de dados
Previsão de atrasos em pedidos.

In [ ]:
import pandas as pd

try:
    from ydata_profiling import ProfileReport
    _profiling_available = True
except ImportError:
    _profiling_available = False
    print("ydata_profiling indisponível (conflito de dependência). O profiling será ignorado.")

df = pd.read_csv('data/amostra.csv')

## Limpeza e tratamento de dados

In [ ]:
import re
import unicodedata

# Convert date columns used for feature engineering
for col in ['dt_pagamento_pedido', 'dt_despacho_pedido', 'dt_previsao_entrega_cliente', 'dt_criacao']:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

# Existing derived features
df['dias_gastos_cd'] = (df['dt_despacho_pedido'] - df['dt_pagamento_pedido']).dt.days
df['dias_restantes_prazo'] = (df['dt_previsao_entrega_cliente'] - df['dt_pagamento_pedido']).dt.days

# Extract dispatch hour BEFORE dropping the column
if 'hr_despacho_pedido' in df.columns:
    df['hora_despacho'] = pd.to_datetime(
        df['hr_despacho_pedido'], format='%H:%M:%S', errors='coerce'
    ).dt.hour

# Normalize city names: case, accents, punctuation and extra spaces
if 'cidade_destinatario' in df.columns:
    def normalize_city(value):
        if pd.isna(value):
            return pd.NA
        text = str(value).strip().lower()
        text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('ascii')
        text = re.sub(r'[^a-z0-9\s]', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text.upper()

    df['cidade_destinatario'] = df['cidade_destinatario'].apply(normalize_city).astype('string')

# Binary encode delivery performance
if 'tp_performance_entrega' in df.columns:
    df['tp_performance_entrega'] = (
        df['tp_performance_entrega']
        .astype('string')
        .str.strip()
        .map({
            'Entregue no Prazo': 1,
            'Fora do Prazo': 0
        })
        .astype('Int64')
    )

df = df.drop(
    columns=[
        'row_id',
        'hr_despacho_pedido',
        'dt_entrega_pedido',
        'hr_entrega_pedido',
        'flg_existem_ocorrencias'
    ],
    errors='ignore'
 )

categorical_features = ['uf', 'grp_transportadora', 'cidade_destinatario', 'tp_praca', 'des_unidade_negocio', 'des_cd_origem']

for col in categorical_features:
    df[col] = df[col].astype('category')

df.head(10)

## Feature Engineering

In [ ]:
import numpy as np

# --- 1. Temporal features from dispatch date ---
df['dia_semana_despacho'] = df['dt_despacho_pedido'].dt.dayofweek
df['mes_despacho']        = df['dt_despacho_pedido'].dt.month
# float32 handles NaT-derived NaN natively (unlike int32)
df['semana_ano']          = df['dt_despacho_pedido'].dt.isocalendar().week.astype('float32')

# Weekend dispatch flag (Sat=5, Sun=6): carrier networks are thinner on weekends
df['is_fds_despacho']   = (df['dia_semana_despacho'] >= 5).astype('int8')

# High-demand season: November (Black Friday) and December (Christmas)
df['is_alta_temporada'] = df['mes_despacho'].isin([11, 12]).astype('int8')

# Dispatch shift — earlier dispatches tend to have better same-day processing
if 'hora_despacho' in df.columns:
    df['turno_despacho'] = pd.cut(
        df['hora_despacho'],
        bins=[-1, 5, 11, 17, 23],
        labels=['Madrugada', 'Manha', 'Tarde', 'Noite']
    ).astype('category')

# --- 2. Derived date-ratio features ---
df['dias_criacao_pagamento'] = (df['dt_pagamento_pedido'] - df['dt_criacao']).dt.days

# Days the carrier has from dispatch to expected delivery
df['prazo_apos_despacho'] = (
    df['dt_previsao_entrega_cliente'] - df['dt_despacho_pedido']
).dt.days

# Fraction of total deadline already consumed inside the CD (>1 = deadline already blown)
df['ratio_cd_prazo'] = (
    df['dias_gastos_cd'] / df['dias_restantes_prazo'].replace(0, np.nan)
).clip(0, 2)

# Net delivery margin: negative = deadline blown before dispatch
df['margem_entrega'] = df['prazo_apos_despacho'] - df['dias_gastos_cd']

# --- Summary ---
new_cols = [
    'dia_semana_despacho', 'mes_despacho', 'semana_ano',
    'is_fds_despacho', 'is_alta_temporada',
    'dias_criacao_pagamento', 'prazo_apos_despacho',
    'ratio_cd_prazo', 'margem_entrega',
]
if 'hora_despacho' in df.columns:
    new_cols = ['hora_despacho', 'turno_despacho'] + new_cols

print(f'Novas features adicionadas ({len(new_cols)}):')
print(df[new_cols].describe(include='all').T[['count', 'mean', 'min', 'max']].to_string())

## Profiling de dados

In [ ]:
if _profiling_available:
    profile = ProfileReport(df, title="Profiling Report")
    html = profile.to_html()
    output_file = 'report.html'
    with open(output_file, 'w') as f:
        f.write(html)
    print(f"Relatório salvo em {output_file}")
else:
    print("Profiling ignorado — instale uma versão compatível de numba/numpy para habilitar.")

In [ ]:
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

# ---------------------------------------------------------
# Step 3: Split Features (X) and Target (y)
# ---------------------------------------------------------
selected_features = [
    # Original features
    'cidade_destinatario',
    'uf',
    'grp_transportadora',
    'dt_previsao_entrega_cliente',
    'dt_criacao',
    'dt_pagamento_pedido',
    'tp_praca',
    'des_unidade_negocio',
    'des_cd_origem',
    'dias_gastos_cd',
    'dias_restantes_prazo',
    # Feature engineering — temporal
    'hora_despacho',
    'turno_despacho',
    'dia_semana_despacho',
    'mes_despacho',
    'semana_ano',
    'is_fds_despacho',
    'is_alta_temporada',
    # Feature engineering — ratios
    'dias_criacao_pagamento',
    'prazo_apos_despacho',
    'ratio_cd_prazo',
    'margem_entrega',
]

available_features = [col for col in selected_features if col in df.columns]
missing_features = [col for col in selected_features if col not in df.columns]

if missing_features:
    print('Missing columns (ignored):', missing_features)

X = df[available_features].copy()
y = df['tp_performance_entrega']

# Remove rows with missing target
valid_mask = y.notna()
X = X.loc[valid_mask].copy()
y = y.loc[valid_mask].astype('int32').copy()

print('Target distribution (full data):')
print(y.value_counts())
print(y.value_counts(normalize=True).round(4))

# Convert date columns to numeric representation (ordinal days)
date_cols = ['dt_previsao_entrega_cliente', 'dt_criacao', 'dt_pagamento_pedido']
for col in [c for c in date_cols if c in X.columns]:
    X[col] = pd.to_datetime(X[col], errors='coerce')
    X[col] = X[col].map(lambda x: x.toordinal() if pd.notna(x) else np.nan).astype('float32')

# Encode categorical columns as numeric codes
categorical_cols = [
    'cidade_destinatario',
    'uf',
    'grp_transportadora',
    'tp_praca',
    'des_unidade_negocio',
    'des_cd_origem',
    'turno_despacho',
]
for col in [c for c in categorical_cols if c in X.columns]:
    X[col] = X[col].astype('category').cat.codes.replace(-1, np.nan).astype('float32')

# Drop any remaining unsupported column types
unsupported_cols = X.select_dtypes(include=['object', 'datetime64[ns]', 'datetimetz']).columns
if len(unsupported_cols) > 0:
    print('Dropping unsupported columns:', list(unsupported_cols))
X = X.drop(columns=unsupported_cols, errors='ignore')

print('Training columns:', list(X.columns))

# Split into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('\nTarget distribution (train before fit):')
print(y_train.value_counts())
print(y_train.value_counts(normalize=True).round(4))

# ---------------------------------------------------------
# Target Encoding (fitted only on training data to avoid leakage)
# Smoothed mean: shrinks small-sample categories toward the global mean
# ---------------------------------------------------------
te_cols_map = {
    'grp_transportadora': 'te_transportadora',
    'uf':                 'te_uf',
    'des_cd_origem':      'te_cd_origem',
    'tp_praca':           'te_praca',
}
smoothing_k  = 10  # higher = more shrinkage toward global mean
global_mean  = float(y_train.mean())

# Reference to original string categories in df (before label encoding)
df_te = df.loc[valid_mask, list(te_cols_map.keys())].copy()
for col in te_cols_map:
    if col in df_te.columns:
        df_te[col] = df_te[col].astype('string')

for src_col, new_col in te_cols_map.items():
    if src_col not in df_te.columns:
        continue
    train_cats = df_te.loc[X_train.index, src_col]
    agg = (
        pd.DataFrame({'cat': train_cats.values, 'y': y_train.values})
        .groupby('cat')['y']
        .agg(['mean', 'count'])
    )
    agg['smoothed'] = (
        (agg['count'] * agg['mean'] + smoothing_k * global_mean)
        / (agg['count'] + smoothing_k)
    )
    mapping = agg['smoothed']
    X_train[new_col] = df_te.loc[X_train.index, src_col].map(mapping).fillna(global_mean).astype('float32').values
    X_test[new_col]  = df_te.loc[X_test.index,  src_col].map(mapping).fillna(global_mean).astype('float32').values

print('Target encoding columns added:', list(te_cols_map.values()))

# ---------------------------------------------------------
# Step 4: Initialize and Train the Model
# ---------------------------------------------------------
model = lgb.LGBMClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42,
    is_unbalance=True
)

model.fit(X_train, y_train)

# ---------------------------------------------------------
# Step 5: Make Predictions (The "Risk Score")
# ---------------------------------------------------------
predictions_binary = model.predict(X_test)
predictions_proba  = model.predict_proba(X_test)[:, 1]

# ---------------------------------------------------------
# Step 6: Evaluate Model Quality
# ---------------------------------------------------------
accuracy  = accuracy_score(y_test, predictions_binary)
precision = precision_score(y_test, predictions_binary, zero_division=0)
recall    = recall_score(y_test, predictions_binary, zero_division=0)
f1        = f1_score(y_test, predictions_binary, zero_division=0)
roc_auc   = roc_auc_score(y_test, predictions_proba)
cm        = confusion_matrix(y_test, predictions_binary)

print('=== Model Metrics ===')
print(f'Accuracy : {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall   : {recall:.4f}')
print(f'F1-score : {f1:.4f}')
print(f'ROC-AUC  : {roc_auc:.4f}')
print('\nConfusion Matrix:')
print(cm)
print('\nClassification Report:')
print(classification_report(y_test, predictions_binary, zero_division=0))

## Otimização de Hiperparâmetros

In [ ]:
# pip install optuna  (if needed)
import optuna
from sklearn.model_selection import train_test_split as _tts

optuna.logging.set_verbosity(optuna.logging.WARNING)

# Internal validation split used only during the search
X_tr, X_val, y_tr, y_val = _tts(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

def objective(trial):
    params = {
        # Learning dynamics
        'learning_rate'     : trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'n_estimators'      : 1000,   # controlled by early stopping

        # Tree complexity
        'num_leaves'        : trial.suggest_int('num_leaves', 31, 255),
        'max_depth'         : trial.suggest_int('max_depth', 4, 10),
        'min_child_samples' : trial.suggest_int('min_child_samples', 20, 300),

        # Regularization
        'subsample'         : trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree'  : trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha'         : trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda'        : trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),

        # Class imbalance — replaces is_unbalance=True
        # < 1 → downweights majority class (on-time) → focuses more on delays
        'scale_pos_weight'  : trial.suggest_float('scale_pos_weight', 0.01, 0.5),

        'random_state': 42,
        'verbose'     : -1,
    }

    clf = lgb.LGBMClassifier(**params)
    clf.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=False),
            lgb.log_evaluation(period=-1),
        ],
    )

    proba = clf.predict_proba(X_val)[:, 1]
    return roc_auc_score(y_val, proba)


study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f'\nMelhor ROC-AUC (validação interna): {study.best_value:.4f}')
print('\nMelhores hiperparâmetros encontrados:')
for k, v in study.best_params.items():
    print(f'  {k:25s}: {v}')

In [ ]:
# ---------------------------------------------------------
# Treinar modelo final com os melhores hiperparâmetros
# ---------------------------------------------------------
best_params = study.best_params.copy()
best_params.update({'n_estimators': 1000, 'random_state': 42, 'verbose': -1})

best_model = lgb.LGBMClassifier(**best_params)
best_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=False),
        lgb.log_evaluation(period=-1),
    ],
)

best_pred_binary = best_model.predict(X_test)
best_pred_proba  = best_model.predict_proba(X_test)[:, 1]

print('=== Modelo Otimizado (threshold = 0.5) ===')
print(f'Accuracy : {accuracy_score(y_test, best_pred_binary):.4f}')
print(f'Precision: {precision_score(y_test, best_pred_binary, zero_division=0):.4f}')
print(f'Recall   : {recall_score(y_test, best_pred_binary, zero_division=0):.4f}')
print(f'F1-score : {f1_score(y_test, best_pred_binary, zero_division=0):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y_test, best_pred_proba):.4f}')
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, best_pred_binary))
print('\nClassification Report:')
print(classification_report(y_test, best_pred_binary, zero_division=0))

# ---------------------------------------------------------
# Ajuste de threshold — maximiza F1 da classe 0 (atrasos)
# ---------------------------------------------------------
thresholds = np.arange(0.05, 0.90, 0.01)
f1_scores_class0 = [
    f1_score(y_test, (best_pred_proba >= t).astype(int), pos_label=0, zero_division=0)
    for t in thresholds
]

best_threshold = thresholds[np.argmax(f1_scores_class0)]
best_pred_thresh = (best_pred_proba >= best_threshold).astype(int)

print(f'\n=== Modelo Otimizado (threshold = {best_threshold:.2f}) ===')
print(f'F1 classe 0 com threshold padrão (0.50): {f1_score(y_test, best_pred_binary, pos_label=0, zero_division=0):.4f}')
print(f'F1 classe 0 com threshold ótimo  ({best_threshold:.2f}): {max(f1_scores_class0):.4f}')
print('\nClassification Report com threshold otimizado:')
print(classification_report(y_test, best_pred_thresh, zero_division=0))

# Atualiza model e variáveis para as células seguintes
model           = best_model
predictions_binary = best_pred_thresh
predictions_proba  = best_pred_proba

In [5]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))
lgb.plot_importance(model, ax=ax)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('Saved: feature_importance.png')

Saved: feature_importance.png


In [6]:
# ---------------------------------------------------------
# Step 7: View the Results
# ---------------------------------------------------------
results_df = X_test.copy()
results_df['probabilidade_atraso'] = predictions_proba
results_df['risco_semaforo'] = pd.cut(
    results_df['probabilidade_atraso'],
    bins=[-0.1, 0.3, 0.7, 1.1],
    labels=['🟢 Verde', '🟡 Amarelo', '🔴 Vermelho']
)
print(results_df[['probabilidade_atraso', 'risco_semaforo']].head(20))

        probabilidade_atraso risco_semaforo
406605              0.742474     🔴 Vermelho
445965              0.738001     🔴 Vermelho
456353              0.665927      🟡 Amarelo
88709               0.425970      🟡 Amarelo
355086              0.325110      🟡 Amarelo
391410              0.795133     🔴 Vermelho
41017               0.790890     🔴 Vermelho
256245              0.260306        🟢 Verde
382323              0.630805      🟡 Amarelo
31770               0.525823      🟡 Amarelo
333940              0.792224     🔴 Vermelho
138271              0.680764      🟡 Amarelo
192752              0.636761      🟡 Amarelo
21996               0.602370      🟡 Amarelo
129463              0.565630      🟡 Amarelo
88752               0.373290      🟡 Amarelo
59484               0.476767      🟡 Amarelo
26396               0.446651      🟡 Amarelo
346042              0.304918      🟡 Amarelo
231106              0.764845     🔴 Vermelho


In [7]:
import joblib

# Build categorical mappings from training data
categorical_mappings = {}
for col in [c for c in categorical_cols if c in df.columns]:
    cats = pd.Series(df.loc[valid_mask, col].astype("string").dropna().unique()).sort_values().tolist()
    categorical_mappings[col] = {v: i for i, v in enumerate(cats)}

bundle = {
    "model": model,
    "selected_features": selected_features,
    "date_cols": ["dt_previsao_entrega_cliente", "dt_criacao", "dt_pagamento_pedido"],
    "categorical_cols": [c for c in categorical_cols if c in selected_features],
    "categorical_mappings": categorical_mappings,
    "threshold": 0.5
}

joblib.dump(bundle, "model_bundle.joblib")
print("Saved model_bundle.joblib")

Saved model_bundle.joblib


In [8]:
df[df['tp_performance_entrega'] == 0]

,cod_pedido,cidade_destinatario,uf,grp_transportadora,dt_despacho_pedido,dt_previsao_entrega_cliente,dt_criacao,dt_pagamento_pedido,tp_praca,des_unidade_negocio,des_cd_origem,qtd_dias_tat,tp_performance_entrega,dias_gastos_cd,dias_restantes_prazo
480776,123922798-1,SAO PAULO,SP,Transportadora 5,2023-09-20,2023-09-20,2023-09-17,2023-09-17,Capital,Multi,PR-Campina G. Sul,4.0,0,3.0,3.0
480777,127715717-1,ARACAJU,SE,Transportadora 2,2023-11-28,2023-12-07,2023-11-26,2023-11-26,Capital,Mono,PR-Campina G. Sul,12.0,0,2.0,11.0
480778,122505811-1,PORTO ALEGRE,RS,Transportadora 4,2023-07-31,2023-08-02,2023-07-31,2023-07-29,Capital,Multi,PR-Campina G. Sul,4.0,0,2.0,4.0
480779,127406554,BELO HORIZONTE,MG,Transportadora 3,2023-11-27,2023-12-01,2023-11-24,2023-11-24,Capital,Multi,PR-Campina G. Sul,9.0,0,3.0,7.0
480780,125203829,ARAPONGAS,PR,Transportadora 4,2023-10-31,2023-11-01,2023-10-27,2023-10-27,Interior,Mono,PR-Campina G. Sul,4.0,0,4.0,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
503527,124807656-1,NITEROI,RJ,Transportadora 3,2023-10-16,2023-10-18,2023-10-14,2023-10-14,Reg. Metropolitana,Mono,SP-Registro,4.0,0,2.0,4.0
503528,127769981,PARNAMIRIM,RN,Transportadora 1,2023-11-27,2023-12-11,2023-11-26,2023-11-26,Interior,Mono,SP-Registro,NaN,0,1.0,15.0
503529,121957602-1,CANOAS,RS,Transportadora 4,2023-07-11,2023-07-12,2023-07-11,2023-07-09,Capital,Multi,PR-Campina G. Sul,4.0,0,2.0,3.0
503530,123101228-1,GUARULHOS,SP,Transportadora 1,NaT,2023-08-23,2023-08-19,2023-08-19,Reg. Metropolitana,Mono,SP-Registro,4.0,0,NaN,4.0
